In [105]:
import os
import sys
PATH = os.getcwd()
DIR_DATA = PATH + '{0}data{0}tass{0}'.format(os.sep)
sys.path.append(PATH) if PATH not in list(sys.path) else None
DIR_DATA


'/content/data/tass/'

In [106]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from collections import Counter
from sklearn import preprocessing
from sklearn.preprocessing import LabelEncoder
from logic.text_processing import TextProcessing
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from imblearn.over_sampling import RandomOverSampler

#agregamos el tfidfvectorizer para cambiar la forma de vectorizar las palabras
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import StratifiedKFold,train_test_split, cross_val_score, ShuffleSplit,StratifiedShuffleSplit, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import classification_report, confusion_matrix, recall_score, log_loss
from sklearn.metrics import f1_score, accuracy_score, precision_score

#!pip install -U spacy # solo si no se tiene la libreria
#!python -m spacy download en_core_web_sm #ni el modelo


#!pip install emoji # Ejecutar si no esta instalada la libreria
import emoji
import re

In [107]:
tp = TextProcessing()
le = LabelEncoder()

Error load_sapcy: [E050] Can't find model 'es_core_news_sm'. It doesn't seem to be a Python package or a valid path to a data directory.


In [108]:
data_train = pd.read_csv(DIR_DATA + 'tass2018_es_train.csv', sep=',')
data_train[:5]

,tweetid,user,content,date,lang,sentiment/polarity/value
0,768213876278165504,OnceBukowski,-Me caes muy bien \r\n-Tienes que jugar más pa...,2016-08-23 22:30:35,es,NONE
1,768213567418036224,anahorxn,@myendlesshazza a. que puto mal escribo\r\n\r\...,2016-08-23 22:29:21,es,N
2,768212591105703936,martitarey13,@estherct209 jajajaja la tuya y la d mucha gen...,2016-08-23 22:25:29,es,N
3,768221670255493120,endlessmilerr,Quiero mogollón a @AlbaBenito99 pero sobretodo...,2016-08-23 23:01:33,es,P
4,768221021300264964,JunoWTFL,Vale he visto la tia bebiendose su regla y me ...,2016-08-23 22:58:58,es,N


In [109]:
data_test = pd.read_csv(DIR_DATA + 'tass2018_es_test.csv', sep=',')
data_test[:5]

,tweetid,user,content,date,lang,sentiment/polarity/value
0,770976639173951488,noseashetero,@noseashetero 1000/10 de verdad a ti que voy a...,2016-08-31 13:28:49,es,P
1,771092421866389508,Templelx,@piscolabisaereo @HistoriaNG @SPosteguillo las...,2016-08-31 21:08:54,es,P
2,771092111429083136,esskuu94,"Al final han sido 3h Bueno, mañana tengo fies...",2016-08-31 21:07:40,es,P
3,771092070572449796,__ariadna9,@Jorge_Ruiz14 yo no tengo tiempo para esas cos...,2016-08-31 21:07:30,es,N
4,771094192508600320,_cristtina15_,@_MissChaotic_ ves ese brillo? es un coso que ...,2016-08-31 21:15:56,es,N


In [110]:
#metodo para combinar las clases de none y neutral en una sola clase
# buscando aumentar el numero total de datos en la clase y evitamos realizar un gran sobremuestreo
def agrupar_sentimientos(sentimiento):
    if sentimiento == 'P': return 'P'
    if sentimiento == 'N': return 'N'
    return 'OTHERS'

data_train['sentiment_agrupado'] = data_train['sentiment/polarity/value'].apply(agrupar_sentimientos)
data_test['sentiment_agrupado'] = data_test['sentiment/polarity/value'].apply(agrupar_sentimientos)

In [111]:
def limpiar_tuit(texto):
    if not isinstance(texto, str):
        return ""

    # 1. Convertir Emojis a texto en español (ej. 😊 -> :cara_sonriendo_con_ojos_corazón:)
    texto = emoji.demojize(texto, language='es')

    # Reemplazar los dos puntos y guiones bajos que genera 'demojize' por espacios
    texto = texto.replace(':', ' ').replace('_', ' ')

    # 2. Eliminar
    texto = re.sub(r'http\S+|www\S+|https\S+', '', texto, flags=re.MULTILINE)

    # 3. Eliminar menciones a usuarios (@usuario)
    texto = re.sub(r'\@\w+', '', texto)

    # 4. Mantener los hashtags pero quitar el símbolo '#'
    texto = texto.replace('#', '')

    # 5. Reducir caracteres repetidos (ej. "jajajajaja" -> "jajaja", "hooooolaaa" -> "hoolaa")
    texto = re.sub(r'(.)\1{2,}', r'\1\1', texto)

    # 6. Eliminar saltos de línea (\r\n) y espacios extra
    texto = re.sub(r'\s+', ' ', texto).strip()

    # Opcional: convertir a minúsculas
    texto = texto.lower()

    return texto

In [112]:
# Aplicamos la limpieza a la columna de contenido
data_train['clean_content'] = data_train['content'].apply(limpiar_tuit)

# Ahora pasamos el texto limpio a tu procesador (spacy, etc.)
x_train = [tp.transformer(row) for row in data_train['clean_content'].tolist()]

# Mantenemos las etiquetas
y_train = data_train['sentiment_agrupado']
print("Train sizes:", len(x_train), len(y_train))

Train sizes: 1008 1008


In [113]:
# Aplicamos la limpieza al test
data_test['clean_content'] = data_test['content'].apply(limpiar_tuit)

x_test = [tp.transformer(row) for row in data_test['clean_content'].tolist()]

y_test = data_test['sentiment_agrupado']
print("Test sizes:", len(x_test), len(y_test))

Test sizes: 506 506


In [114]:
vectorizer = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1,3),
    min_df=3,
    max_df=0.85,
    max_features = 5000,
    sublinear_tf=True,
    use_idf=True
)

In [115]:
x_train = vectorizer.fit_transform(x_train)


x_train.toarray()


array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [116]:
x_test = vectorizer.transform(x_test)

In [117]:
x_test.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [118]:
print('**Sample train:', sorted(Counter(y_train).items()))

**Sample train: [('N', 418), ('OTHERS', 272), ('P', 318)]


In [119]:
print('**Sample test:', sorted(Counter(y_test).items()))

**Sample test: [('N', 219), ('OTHERS', 131), ('P', 156)]


In [120]:

# 1. Definimos una lista de valores para el parámetro C
parametros = {'C': [0.01, 0.1, 0.5, 1, 5, 10]}

# 2. Instanciamos el modelo base
svm_base = LinearSVC(class_weight='balanced', max_iter=3000, random_state=42)

# 3. Configuramos la búsqueda en cuadrícula (GridSearch) para que busque el mejor valor de c
grid_search = GridSearchCV(svm_base, parametros, cv=5, scoring='f1_macro', n_jobs=-1)

# 4. Entrenamos con TODO el dataset de entrenamiento
grid_search.fit(x_train, y_train)

print(f"Mejor parámetro C encontrado: {grid_search.best_params_}")

# 5. Guardamos el modelo ganador
kfold = grid_search.best_estimator_

Mejor parámetro C encontrado: {'C': 0.5}


In [121]:
ros_train = RandomOverSampler(random_state=1000)
x_train, y_train = ros_train.fit_resample(x_train, y_train)

In [122]:
print('**OverSample train:', sorted(Counter(y_train).items()))

**OverSample train: [('N', 418), ('OTHERS', 418), ('P', 418)]


In [123]:
ros_test = RandomOverSampler(random_state=1000)
x_test, y_test = ros_test.fit_resample(x_test, y_test)

In [124]:
print('**OverSample test:', sorted(Counter(y_test).items()))

**OverSample test: [('N', 219), ('OTHERS', 219), ('P', 219)]


In [125]:
softmax = LinearSVC(
    C=0.5,
    class_weight='balanced',
    max_iter=2000,
    random_state=42
)

In [126]:
accuracies_scores = []
recalls_scores = []
precisions_scores = []
f1_scores = []

In [127]:

# 1. Inicializamos el particionador (K-Fold) con 5 cortes
k_fold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

y_train_array = np.array(y_train)

# Limpiamos las listas por si corres la celda varias veces
accuracies_scores = []
recalls_scores = []
precisions_scores = []
f1_scores = []

# 3. Tu ciclo for, ahora sí con k_fold y y_train_array
for train_index, test_index in k_fold.split(x_train, y_train_array):

    # Extraemos los datos de entrenamiento para este fold
    data_train = x_train[train_index]
    target_train = y_train_array[train_index]

    # Extraemos los datos de prueba para este fold
    data_test = x_train[test_index]
    target_test = y_train_array[test_index]

    # Entrenamos y predecimos
    softmax.fit(data_train, target_train)
    predict = softmax.predict(data_test)

    # Guardamos las métricas
    accuracies_scores.append(accuracy_score(target_test, predict))
    recalls_scores.append(recall_score(target_test, predict, average='macro'))
    precisions_scores.append(precision_score(target_test, predict, average='weighted', zero_division=0))
    f1_scores.append(f1_score(target_test, predict, average='weighted'))

print("¡Validación cruzada terminada!")

¡Validación cruzada terminada!


In [128]:
average_recall = round(np.mean(recalls_scores) * 100, 2)
average_precision = round(np.mean(precisions_scores) * 100, 2)
average_f1 = round(np.mean(f1_scores) * 100, 2)
average_accuracy = round(np.mean(accuracies_scores) * 100, 2)

In [129]:
# Usamos el modelo optimizado para predecir directamente todo el conjunto de prueba
y_predict = kfold.predict(x_test)


classification = classification_report(y_test, y_predict)
confusion = confusion_matrix(y_test, y_predict)

In [130]:
output_result = {'F1-score': average_f1, 'Accuracy': average_accuracy, 'Recall': average_recall,
                 'Precision': average_precision, 'Classification Report\n': classification,
                 'Confusion Matrix\n': confusion}

In [131]:
for item, val in output_result.items():
    print('{0} {1}'.format(item, val))

F1-score 61.48
Accuracy 61.49
Recall 61.5
Precision 61.66
Classification Report
               precision    recall  f1-score   support

           N       0.48      0.62      0.54       219
      OTHERS       0.41      0.32      0.36       219
           P       0.56      0.53      0.54       219

    accuracy                           0.49       657
   macro avg       0.49      0.49      0.48       657
weighted avg       0.49      0.49      0.48       657

Confusion Matrix
 [[136  48  35]
 [ 94  70  55]
 [ 53  51 115]]
